# Автоматическое ревью pull request'ов в Azure DevOps Server

**Автор**: auto_git_review (Rail-Akhm)
**Стек**: Python 3.10+ · Apache Airflow · Azure DevOps Server (on-prem) · LiteLLM / qwen3

---

## Цель

Разобрать и показать, как устроен production-пайплайн автоматического ревью
открытых pull request'ов: от нахождения PR в корпоративном ALM через сбор diff и
истории файлов до вердикта локальной LLM и (опционально) комментария к PR.
Ноутбук повторяет структуру `similarity_search/edm_research.ipynb`: весь
production-код (`config.py`, `alm.py`, `llm.py`, `prompt.py`, `review.py`)
встроен в клетки **дословно** — файл-источник указан в заголовке каждой вставки.

## Целевые свойства пайплайна

| Свойство | Ограничение / ожидание |
|---|---|
| Контекст одного вызова LLM | ≤ ~8–10k токенов: батч ≤ `MAX_BATCH_CHARS = 30000` симв. |
| Полный текст одного файла в промпте | ≤ `MAX_CHANGE_CHARS = 12000` симв. (модель видит весь файл, а не фрагменты diff) |
| Дублей комментариев на повторных запусках | 0 — dedup по маркеру `auto_git_review` до вызова LLM |
| Комментариев на один PR | 1 общий thread (без привязки к строке) |
| Защита от гигантских PR | больше `MAX_BATCHES = 20` батчей → резюме «требуется ручное ревью» без вызова LLM |

## Проверяемые гипотезы (инварианты)

| № | Гипотеза | Где проверяется |
|---|---|---|
| **H1** | Проверка dedup-маркера по тексту (а не по автору) до вызова LLM гарантирует отсутствие повторного ревью и не трогает комментарии реального пользователя | Раздел 4 |
| **H2** | Батчевание держит каждый вызов LLM в лимите контекста; обрезка `MAX_CHANGE_CHARS`/`MAX_DIFF_CHARS` защищает от переполнения | Раздел 5 |
| **H3** | Агрегация вердиктов батчей по приоритету `request_changes > comment > approve` корректна; JSON-ответ модели разбирается толерантно | Раздел 6 |
| **H4** | Лимит `MAX_BATCHES` не даёт прогнать через LLM слишком большой PR — формируется резюме «требуется ручное ревью» | Раздел 7 |

---
## 1. Постановка задачи

### 1.1. Проблема (AS IS)

Первичное ревью pull request'ов инженеры выполняют вручную. Рутина: открыть PR,
проверить логику SQL/скриптов, вспомнить историю правок файла. При этом в
корпоративном контуре (локальный ALM + локальная LLM) нет внешних сервисов — всё
должно работать внутри периметра.

| | Ручное ревью | Автоматическое (LLM) |
|---|---|---|
| Скорость | минуты–часы на PR | минуты, без участия человека |
| Память о прошлых правках файла | зависит от ревьюера | последние 5 коммитов по файлу |
| Масштаб | тяжело на больших PR | батчевое ревью частями |
| Качество | экспертное | «второй взгляд» на корректность |

### 1.2. Архитектура решения

Пайплайн (одна таска Airflow на репозиторий, каждый репозиторий — свой проект ALM
и свой тип кодовой базы с отдельным промптом):

```
┌───────────────┐   ┌────────────────────┐   ┌──────────────┐   ┌───────────────┐
│ ALM: открытые │──▶│ Сбор изменений PR:  │──▶│ Батчи файлов │──▶│ LLM по каждому│
│ PR репозитория│   │ diff + полное       │   │ (≤30k симв.) │   │ батчу:        │
│               │   │ содержимое +        │   │              │   │ verdict/      │
│               │   │ история 5 коммитов  │   │              │   │ summary/...   │
└───────────────┘   └────────────────────┘   └──────────────┘   └───────┬───────┘
      ▲                                                                  │
      │  (повторные запуски)                                             ▼
┌───────────────┐   ┌────────────────────┐   ┌───────────────────────────────┐
│ PR уже имеет  │──▶│ Пропустить (dedup) │   │ Агрегация:                    │
│ комментарий с │   │ до вызова LLM      │   │ request_changes>comment>approve│
│ маркером      │   │                    │   │ + markdown-комментарий в PR    │
│ auto_git_review│  │                    │   │ (опционально)                  │
└───────────────┘   └────────────────────┘   └───────────────────────────────┘
```

Детали производственной оркестрации — в ДАГе `dags/prj_example/auto_git_review_dag.py`
(6 репозиториев в двух проектах ALM: `ExampleProject1` и `ExampleProject2`). Секреты
берутся из Airflow Connections `api_git` и `llm_server`.

> **Безопасность**: ноутбук ничего не пишет в ALM — флаг постинга комментариев
> (`post_comments`) везде выключен. Живые вызовы (разделы 2 и 8) — только чтение.

---
## 2. Окружение и данные

### 2.1. Импорты и конфигурация

Параметры читаются из переменных окружения (как в `config.py`). Несекретные
URL-дефолты корпоративного контура заданы прямо в коде, секреты (`AZURE_DEVOPS_PAT`,
`LLM_API_KEY`) подставляются из `.env` (см. `.env.example`).

> **Примечание.** Все адреса эндпоинтов (ALM, LLM) и имена проектов/репозиториев в ноутбуке — **примеры** (`example.local`, `ExampleProject1`, …). Реальные значения задаются на сервере развёртывания (.env / Connections Airflow) и в этом материале не раскрываются.

In [1]:
# === Модуль config.py — конфигурация из env (production-код, дословно) ===

# Для ноутбука: грузим .env из текущей папки (аналог строки конфига в edm_research.ipynb).
try:
    from dotenv import load_dotenv
    load_dotenv(".env")
except ImportError:
    pass



"""Конфигурация: читается из переменных окружения.

В Airflow переменные окружения подставляются из Connections (см. ДАГ):
  - api_git    -> AZURE_DEVOPS_URL, AZURE_DEVOPS_PAT
  - llm_server -> LLM_URL, LLM_API_KEY
"""

import os
from dataclasses import dataclass

import urllib3

VERIFY_SSL = os.environ.get("VERIFY_SSL", "false").lower() in ("1", "true", "yes", "on")

if not VERIFY_SSL:
    # Самоподписанные сертификаты на корпоративных хостах — осознанно выключаем проверку.
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Корпоративные константы (НЕ секреты) — дефолты, чтобы задавать только секреты.
# azure_url — база коллекции (БЕЗ проекта); проект задаётся отдельно (azure_project),
# т.к. репозитории могут лежать в разных проектах ALM.
AZURE_URL_DEFAULT = "https://alm.example.local:8080/TFS/ExampleCollection"
AZURE_PROJECT_DEFAULT = "ExampleProject1"
AZURE_REPO_DEFAULT = "example_repo_greenplum"
LLM_URL_DEFAULT = "https://llm.example.local/v1/chat/completions"


@dataclass(frozen=True)
class Settings:
    azure_url: str
    azure_project: str
    azure_pat: str
    azure_repo: str
    api_version: str
    llm_url: str
    llm_api_key: str
    llm_model: str
    post_comments: bool
    verify_ssl: bool = VERIFY_SSL


def _env_bool(name: str, default: str = "false") -> bool:
    return os.environ.get(name, default).lower() in ("1", "true", "yes", "on")


def get_settings() -> Settings:
    return Settings(
        azure_url=os.environ.get("AZURE_DEVOPS_URL", AZURE_URL_DEFAULT).rstrip("/"),
        azure_project=os.environ.get("AZURE_DEVOPS_PROJECT", AZURE_PROJECT_DEFAULT),
        azure_pat=os.environ.get("AZURE_DEVOPS_PAT", "").strip(),
        azure_repo=os.environ.get("AZURE_DEVOPS_REPO", AZURE_REPO_DEFAULT),
        api_version=os.environ.get("AZURE_DEVOPS_API_VERSION", "6.1-preview"),
        llm_url=os.environ.get("LLM_URL", LLM_URL_DEFAULT).rstrip("/"),
        llm_api_key=os.environ.get("LLM_API_KEY", "").strip(),
        llm_model=os.environ.get("LLM_MODEL", "qwen3:latest"),
        post_comments=_env_bool("POST_COMMENTS", "false"),
        verify_ssl=VERIFY_SSL,
    )

# --- Дополнение для ноутбука: помощник для «живых» клеток и маскирование секретов ---
def require_live():
    missing = []
    if not settings.azure_pat:
        missing.append("AZURE_DEVOPS_PAT")
    if not settings.llm_api_key:
        missing.append("LLM_API_KEY")
    if missing:
        raise RuntimeError(
            "Нет доступа к живым эндпоинтам: отсутствуют "
            + ", ".join(missing)
            + ". Заполните .env из .env.example и запустите в корпоративной сети."
        )

def mask(s: str) -> str:
    return (s[:4] + "…" + s[-2:]) if len(s) > 8 else ("***" if s else "")

# Максимум батчей на один PR (зеркало DAGConfiguration.MAX_BATCHES из auto_git_review_dag.py)
MAX_BATCHES = 20

settings = get_settings()
print("ALM URL      :", settings.azure_url)
print("Проект       :", settings.azure_project)
print("Репозиторий  :", settings.azure_repo)
print("API version  :", settings.api_version)
print("LLM endpoint :", settings.llm_url)
print("LLM модель   :", settings.llm_model)
print("AZURE_DEVOPS_PAT :", mask(settings.azure_pat), "| LLM_API_KEY:", mask(settings.llm_api_key))
print("POST_COMMENTS     :", settings.post_comments, "| VERIFY_SSL:", settings.verify_ssl)

ALM URL      : https://alm.example.local:8080/TFS/ExampleCollection
Проект       : ExampleProject1
Репозиторий  : example_repo_greenplum
API version  : 6.1-preview
LLM endpoint : https://llm.example.local/v1/chat/completions
LLM модель   : qwen3:latest
AZURE_DEVOPS_PAT :  | LLM_API_KEY: 
POST_COMMENTS     : False | VERIFY_SSL: False


### 2.2. Реестр репозиториев (как в `DAGConfiguration.REPOSITORIES`)

Оркестрация мульти-репозиторная: репозитории лежат в разных проектах ALM, у
каждого — свой тип кодовой базы и свой промпт.

In [2]:
# === Реестр репозиториев — срез DAGConfiguration.REPOSITORIES (auto_git_review_dag.py) ===
REPOSITORIES = [
    # (проект, репозиторий, тип кодовой базы, промпт)
    ("ExampleProject1", "example_repo_greenplum", "Greenplum SQL",       "review_prompt_greenplum.md"),
    ("ExampleProject1", "example_repo_airflow_etl",     "Airflow ETL",         "review_prompt_airflow_etl.md"),
    ("ExampleProject2",             "example_repo_airflow_orch",           "Airflow (оркестрация)","review_prompt_airflow_orchestration.md"),
    ("ExampleProject2",             "example_repo_clickhouse",                  "ClickHouse SQL",      "review_prompt_clickhouse.md"),
    ("ExampleProject2",             "example_repo_adb",                   "Greenplum SQL",       "review_prompt_greenplum.md"),
    ("ExampleProject2",             "example_repo_postgres",              "PostgreSQL SQL",      "review_prompt_postgres.md"),
]
for project, repo, kind, prompt in REPOSITORIES:
    print(f"{project:22s} | {repo:32s} | {kind:22s} | {prompt}")

ExampleProject1        | example_repo_greenplum           | Greenplum SQL          | review_prompt_greenplum.md
ExampleProject1        | example_repo_airflow_etl         | Airflow ETL            | review_prompt_airflow_etl.md
ExampleProject2        | example_repo_airflow_orch        | Airflow (оркестрация)  | review_prompt_airflow_orchestration.md
ExampleProject2        | example_repo_clickhouse          | ClickHouse SQL         | review_prompt_clickhouse.md
ExampleProject2        | example_repo_adb                 | Greenplum SQL          | review_prompt_greenplum.md
ExampleProject2        | example_repo_postgres            | PostgreSQL SQL         | review_prompt_postgres.md


### 2.3. Клиент ALM — доступ к данным (production `alm.py`)

ALM — источник данных пайплайна (аналог «загрузки CSV» в `edm_research.ipynb`).
Класс `AlmClient` — тонкая обёртка над REST Azure DevOps Server:
Git-эндпоинты project-scoped (`/{{collection}}/{{project}}/_apis/...`), work items —
коллекционные. Аутентификация PAT + `HTTPBasicAuth`, TLS `verify=False`
(самоподписанный корпоративный сертификат).

In [3]:
# === Модуль alm.py — клиент Azure DevOps Server (production-код, дословно) ===
# (убраны только относительные импорты: get_settings уже определён выше)

"""Клиент REST API Azure DevOps Server (on-prem)."""

import requests
from requests.auth import HTTPBasicAuth



class AlmClient:
    def __init__(self, settings=None):
        self.settings = settings or get_settings()
        self.session = requests.Session()
        self.session.auth = HTTPBasicAuth("", self.settings.azure_pat)
        self.session.verify = self.settings.verify_ssl

    def _url(self, path: str) -> str:
        # Git-эндпоинты — project-scoped: /{collection}/{project}/_apis/...
        return f"{self.settings.azure_url}/{self.settings.azure_project}/{path.lstrip('/')}"

    def _url_collection(self, path: str) -> str:
        # Коллекционные эндпоинты (без проекта в пути), напр. WIT.
        return f"{self.settings.azure_url}/{path.lstrip('/')}"

    def _get(self, path: str, params=None) -> dict:
        p = dict(params or {})
        p.setdefault("api-version", self.settings.api_version)
        resp = self.session.get(self._url(path), params=p, timeout=60)
        resp.raise_for_status()
        return resp.json()

    def _get_collection(self, path: str, params=None) -> dict:
        p = dict(params or {})
        p.setdefault("api-version", self.settings.api_version)
        resp = self.session.get(self._url_collection(path), params=p, timeout=60)
        resp.raise_for_status()
        return resp.json()

    def list_open_pull_requests(self, repo: str = None) -> dict:
        """Все открытые PR репозитория."""
        repo = repo or self.settings.azure_repo
        return self._get(
            f"_apis/git/repositories/{repo}/pullrequests",
            params={"searchCriteria.status": "active"},
        )

    def list_completed_pull_requests(self, repo: str = None, top: int = 200) -> dict:
        """Последние завершённые PR (нужны для контекста «5 предыдущих PR»)."""
        repo = repo or self.settings.azure_repo
        return self._get(
            f"_apis/git/repositories/{repo}/pullrequests",
            params={"searchCriteria.status": "completed", "$top": top},
        )

    def get_pull_request(self, pr_id: int, repo: str = None) -> dict:
        repo = repo or self.settings.azure_repo
        return self._get(f"_apis/git/repositories/{repo}/pullrequests/{pr_id}")

    def get_pr_work_items(self, pr_id: int, repo: str = None) -> dict:
        repo = repo or self.settings.azure_repo
        return self._get(
            f"_apis/git/repositories/{repo}/pullrequests/{pr_id}/workitems"
        )

    def get_work_item(self, work_item_id: int) -> dict:
        # WIT — коллекционный эндпоинт (не привязан к проекту).
        return self._get_collection(f"_apis/wit/workitems/{work_item_id}")

    def get_pr_changes(self, source_commit: str, target_commit: str, repo: str = None) -> dict:
        """Низкоуровневый diff между двумя коммитами (diffs/commits).

        ВАЖНО: без ``baseVersionType``/``targetVersionType=commit`` сервер трактует
        SHA как имя ветки и отвечает 404 (GitUnresolvableToCommitException).

        Возвращает ``changes[]``, но вместе с файлами там и ДИРЕКТОРИИ
        (tree-объекты: /db, /db/current, ...). Контента в ответе НЕТ.
        Для списка реально изменённых файлов PR используй get_pr_iteration_changes().
        """
        repo = repo or self.settings.azure_repo
        return self._get(
            f"_apis/git/repositories/{repo}/diffs/commits",
            params={
                "baseVersion": target_commit,
                "baseVersionType": "commit",
                "targetVersion": source_commit,
                "targetVersionType": "commit",
            },
        )

    def get_pr_iterations(self, pr_id: int, repo: str = None) -> dict:
        """Итерации PR. Каждая содержит sourceRefCommit/targetRefCommit/commonRefCommit."""
        repo = repo or self.settings.azure_repo
        return self._get(f"_apis/git/repositories/{repo}/pullRequests/{pr_id}/iterations")

    def get_pr_iteration_changes(self, pr_id: int, iteration_id: int, repo: str = None) -> dict:
        """Точный список изменённых ФАЙЛОВ PR в итерации (changeEntries[]).

        Каждая запись: {changeType: add|edit|delete|..., item: {path, objectId,
        originalObjectId, gitObjectType}}. Контента в ответе НЕТ — его надо
        дотягивать отдельно через get_file_content().
        """
        repo = repo or self.settings.azure_repo
        return self._get(
            f"_apis/git/repositories/{repo}/pullRequests/{pr_id}/iterations/{iteration_id}/changes",
            params={"$top": 2000},
        )

    def get_file_content(self, path: str, version: str, repo: str = None) -> str:
        """Сырое содержимое файла ``path`` на версии ``version`` (SHA коммита).

        Endpoint ``items?includeContent=true`` для одного файла возвращает ТЕКСТ
        файла напрямую (не JSON), поэтому здесь читается resp.text, а не resp.json().
        """
        repo = repo or self.settings.azure_repo
        resp = self.session.get(
            self._url(f"_apis/git/repositories/{repo}/items"),
            params={
                "path": path,
                "includeContent": "true",
                "versionDescriptor.version": version,
                "versionDescriptor.versionType": "commit",
                "api-version": self.settings.api_version,
            },
            timeout=60,
        )
        resp.raise_for_status()
        return resp.text

    def get_threads(self, pr_id: int, repo: str = None) -> dict:
        """Все комментарии (threads) PR. Ответ: {value: [...], count: N}."""
        repo = repo or self.settings.azure_repo
        return self._get(
            f"_apis/git/repositories/{repo}/pullRequests/{pr_id}/threads"
        )

    def create_thread_comment(self, pr_id: int, content: str, repo: str = None) -> dict:
        """Создать общий комментарий (thread) к PR. Возвращает созданный thread.

        commentType=1 (text), status=1 (active), без threadContext — комментарий
        к самому PR, не к конкретному файлу/строке.
        """
        repo = repo or self.settings.azure_repo
        body = {
            "comments": [
                {
                    "parentCommentId": 0,
                    "content": content,
                    "commentType": 1,
                }
            ],
            "status": 1,
        }
        resp = self.session.post(
            self._url(f"_apis/git/repositories/{repo}/pullRequests/{pr_id}/threads"),
            params={"api-version": self.settings.api_version},
            json=body,
            timeout=60,
        )
        resp.raise_for_status()
        return resp.json()

    def get_file_commits(self, path: str, target_version: str, top: int = 5, repo: str = None) -> dict:
        """История коммитов, затрагивавших файл path (до версии target_version)."""
        repo = repo or self.settings.azure_repo
        params = {
            "searchCriteria.itemPath": path,
            "$top": top,
        }
        if target_version:
            params["searchCriteria.itemVersion.version"] = target_version
            params["searchCriteria.itemVersion.versionType"] = "commit"
        return self._get(f"_apis/git/repositories/{repo}/commits", params=params)

print("alm.py: AlmClient готов.")

alm.py: AlmClient готов.


### 2.4. Живой обзор: открытые PR по репозиториям (EDA реальной выгрузки)

Клетка ходит в реальный ALM и показывает «сырую» картину: сколько открытых PR и
кто авторы. **Требует** `.env` с `AZURE_DEVOPS_PAT` и корпоративной сети. Без них
клетка аккуратно пропускается (вывод `SKIP`), остальной ноутбук продолжает работать.

In [4]:
# === 2.4 Живой EDA: открытые PR репозиториев ===
try:
    require_live()
    alm = AlmClient(settings)
    for project, repo, kind, prompt in REPOSITORIES[:3]:          # три примера
        prs = alm.list_open_pull_requests(repo=repo).get("value", [])
        print(f"\n[{project}/{repo}] открытых PR: {len(prs)}")
        for p in prs[:5]:
            author = p.get("createdBy", {}).get("displayName", "?")
            print(f"  #{p['pullRequestId']:>5}  {p.get('title','')[:60]:60s}  {author}")
        if not prs:
            print("  (нет открытых PR)")
except Exception as exc:
    print(f"SKIP 2.4 (живой обзор): {type(exc).__name__}: {exc}")

SKIP 2.4 (живой обзор): RuntimeError: Нет доступа к живым эндпоинтам: отсутствуют AZURE_DEVOPS_PAT, LLM_API_KEY. Заполните .env из .env.example и запустите в корпоративной сети.


---
## 3. Базовые компоненты пайплайна

Три «кита»: клиент LLM (`llm.py`), промпты (`prompt.py` + шаблоны `prompts/*.md`)
и ядро ревью (`review.py`) — сбор контекста файла, батчирование, вызов LLM,
агрегация результата.

### 3.1. Клиент LLM (production `llm.py`)

Тонкая обёртка над OpenAI-совместимым endpoint LiteLLM-шлюза. Модель вынесена в
конфиг — менять без правки кода. Аутентификация заголовком `x-litellm-api-key`.

In [5]:
# === Модуль llm.py — клиент LiteLLM (production-код, дословно) ===
# (убраны только относительные импорты)

"""Клиент локального LLM через LiteLLM-шлюз (OpenAI-совместимый)."""

import requests



class LlmClient:
    def __init__(self, settings=None):
        self.settings = settings or get_settings()

    def chat(self, messages, temperature=None, timeout=600) -> dict:
        """Один вызов chat/completions. Возвращает {content, reasoning}."""
        payload = {
            "model": self.settings.llm_model,
            "messages": messages,
        }
        if temperature is not None:
            payload["temperature"] = temperature

        resp = requests.post(
            self.settings.llm_url,
            verify=self.settings.verify_ssl,
            headers={
                "x-litellm-api-key": self.settings.llm_api_key,
                "Content-Type": "application/json",
            },
            json=payload,
            timeout=timeout,
        )
        resp.raise_for_status()
        data = resp.json()
        message = data["choices"][0]["message"]
        return {
            "content": message.get("content") or "",
            "reasoning": message.get("reasoning_content") or "",
        }

print("llm.py: LlmClient готов.")

llm.py: LlmClient готов.


### 3.2. Промпты (production `prompt.py` + `prompts/*.md`)

В репозитории промпты лежат отдельными файлами `prompts/*.md` — свой под каждый
тип кодовой базы, правится без правки кода. Плейсхолдеры `[[WORK_ITEM]]` и
`[[FILES]]` подставляются через `render_prompt()`. Здесь шаблоны встроены в клетку
дословно (как в `edm_research.ipynb` был встроен production-промпт), чтобы ноутбук
был самодостаточным; функция `load_prompt` в production читает файлы пакета.

In [6]:
# === Промпты: production-шаблоны prompts/*.md, встроены дословно ===
PROMPTS = {
    'review_prompt_airflow_etl.md': """Ты — старший инженер-ревьюер. Проанализируй изменения в pull request и оцени их
корректность.

Кодовая база — ДАГи Apache Airflow, реализующие полноценные ETL-пайплайны
(извлечение, трансформация и загрузка данных).

ПРАВИЛА ЯЗЫКА (строго, нарушение недопустимо):
- Отвечай ТОЛЬКО на русском языке. Все текстовые поля — "summary" и каждый
  "comments[].text" — должны быть на русском.
- Запрещено писать резюме или комментарии на английском (или любом другом языке),
  даже если diff, имена переменных или commit-сообщения на английском.
- Имена схем, таблиц, столбцов, функций и прочие технические идентификаторы
  оставляй без перевода — это код, а не текст.

ПРАВИЛА ПРОТИВ ГАЛЛЮЦИНАЦИЙ:
- Опирайся ТОЛЬКО на предоставленный diff и историю изменений. Не выдумывай код,
  названия схем/таблиц/полей или изменения, которых нет в diff.
- В поле "file" указывай только реальные пути из списка изменённых файлов.
- В поле "line" указывай только строки, реально присутствующие в diff; если точной
  строки нет — не заполняй "line".
- Если замечаний нет, верни пустой массив "comments": []. Не придумывай замечания,
  чтобы «что-то написать».
- Если по данным что-то не ясно — честно напиши об этом в "summary", не додумывай.

Критерии проверки (по убыванию важности):
1. Логические ошибки и баги.
2. Корректность DAG:
   - корректность зависимостей между тасками (отсутствие зацикливаний и битых ссылок);
   - идемпотентность тасок (повторный запуск не должен портить данные);
   - корректная работа с Connections, Hooks, XCom (не передавать через XCom
     большие объёмы данных);
   - обработка ошибок, retries, timeout'ы, корректное завершение.
3. Производительность: распараллеливание, отсутствие лишних вычислений,
   эффективность трансформаций.
4. Безопасность: секреты не должны быть зашиты в код (использовать
   Connections/Secrets), SQL-инъекции, работа с чувствительными данными.
5. Соответствие стилю и конвенциям кода.

Описание связанной задачи (work item):
[[WORK_ITEM]]

Изменённые файлы (полное содержимое + изменения + история предыдущих изменений):
[[FILES]]

Правила анализа:
- Если у файла есть история предыдущих изменений — учитывай её (сложившиеся паттерны, зачем файл менялся).
- Если файл создаётся впервые (истории нет) — оценивай только корректность его кода.

Верни результат строго в формате JSON и ничего больше. Значения "verdict" оставь
латинскими (это фиксированный перечень), а "summary" и "text" пиши по-русски:
{
  "verdict": "approve | request_changes | comment",
  "summary": "краткое резюме ревью на русском",
  "comments": [
    {
      "file": "путь/к/файлу",
      "line": 42,
      "severity": "critical | major | minor | nit",
      "text": "текст комментария на русском"
    }
  ]
}
""",
    'review_prompt_airflow_orchestration.md': """Ты — старший инженер-ревьюер. Проанализируй изменения в pull request и оцени их
корректность.

Кодовая база — ДАГи Apache Airflow, оркестрирующие внешний инструмент
Plus7Formit (запуск и контроль исполнения, без тяжёлой трансформации данных).

ПРАВИЛА ЯЗЫКА (строго, нарушение недопустимо):
- Отвечай ТОЛЬКО на русском языке. Все текстовые поля — "summary" и каждый
  "comments[].text" — должны быть на русском.
- Запрещено писать резюме или комментарии на английском (или любом другом языке),
  даже если diff, имена переменных или commit-сообщения на английском.
- Имена схем, таблиц, столбцов, функций и прочие технические идентификаторы
  оставляй без перевода — это код, а не текст.

ПРАВИЛА ПРОТИВ ГАЛЛЮЦИНАЦИЙ:
- Опирайся ТОЛЬКО на предоставленный diff и историю изменений. Не выдумывай код,
  названия схем/таблиц/полей или изменения, которых нет в diff.
- В поле "file" указывай только реальные пути из списка изменённых файлов.
- В поле "line" указывай только строки, реально присутствующие в diff; если точной
  строки нет — не заполняй "line".
- Если замечаний нет, верни пустой массив "comments": []. Не придумывай замечания,
  чтобы «что-то написать».
- Если по данным что-то не ясно — честно напиши об этом в "summary", не додумывай.

Критерии проверки (по убыванию важности):
1. Логические ошибки и баги.
2. Корректность DAG:
   - корректность вызова внешнего инструмента/API (параметры, endpoint'ы);
   - корректность проверки статуса и обработки результата запуска;
   - идемпотентность запуска (повторный запуск не должен дублировать работу);
   - корректные таймауты, retries, сенсоры.
3. Надёжность: обработка ошибок и частичных сбоев, корректное завершение при отказе.
4. Безопасность: секреты не должны быть зашиты в код, безопасная передача параметров.
5. Соответствие стилю и конвенциям кода.

Описание связанной задачи (work item):
[[WORK_ITEM]]

Изменённые файлы (полное содержимое + изменения + история предыдущих изменений):
[[FILES]]

Правила анализа:
- Если у файла есть история предыдущих изменений — учитывай её (сложившиеся паттерны, зачем файл менялся).
- Если файл создаётся впервые (истории нет) — оценивай только корректность его кода.

Верни результат строго в формате JSON и ничего больше. Значения "verdict" оставь
латинскими (это фиксированный перечень), а "summary" и "text" пиши по-русски:
{
  "verdict": "approve | request_changes | comment",
  "summary": "краткое резюме ревью на русском",
  "comments": [
    {
      "file": "путь/к/файлу",
      "line": 42,
      "severity": "critical | major | minor | nit",
      "text": "текст комментария на русском"
    }
  ]
}
""",
    'review_prompt_clickhouse.md': """Ты — старший инженер-ревьюер. Проанализируй изменения в pull request и оцени их
корректность.

Кодовая база — SQL-скрипты для ClickHouse (колоночная аналитическая СУБД).

ПРАВИЛА ЯЗЫКА (строго, нарушение недопустимо):
- Отвечай ТОЛЬКО на русском языке. Все текстовые поля — "summary" и каждый
  "comments[].text" — должны быть на русском.
- Запрещено писать резюме или комментарии на английском (или любом другом языке),
  даже если diff, имена переменных или commit-сообщения на английском.
- Имена схем, таблиц, столбцов, функций и прочие технические идентификаторы
  оставляй без перевода — это код, а не текст.

ПРАВИЛА ПРОТИВ ГАЛЛЮЦИНАЦИЙ:
- Опирайся ТОЛЬКО на предоставленный diff и историю изменений. Не выдумывай код,
  названия схем/таблиц/полей или изменения, которых нет в diff.
- В поле "file" указывай только реальные пути из списка изменённых файлов.
- В поле "line" указывай только строки, реально присутствующие в diff; если точной
  строки нет — не заполняй "line".
- Если замечаний нет, верни пустой массив "comments": []. Не придумывай замечания,
  чтобы «что-то написать».
- Если по данным что-то не ясно — честно напиши об этом в "summary", не додумывай.

Критерии проверки (по убыванию важности):
1. Логические ошибки и баги.
2. Корректность SQL для ClickHouse:
   - корректный выбор движка таблицы (MergeTree, ReplacingMergeTree,
     SummingMergeTree, AggregatingMergeTree, Distributed, MaterializedView);
   - корректность ключей ORDER BY / PARTITION BY (выбор ключа сортировки);
   - особенности диалекта ClickHouse SQL (отличия от стандартного SQL).
3. Производительность: ключ сортировки, партиционирование, TTL,
   материализованные представления, словари.
4. Безопасность: SQL-инъекции, права доступа, работа с чувствительными данными.
5. Соответствие стилю и конвенциям кода.

Описание связанной задачи (work item):
[[WORK_ITEM]]

Изменённые файлы (полное содержимое + изменения + история предыдущих изменений):
[[FILES]]

Правила анализа:
- Если у файла есть история предыдущих изменений — учитывай её (сложившиеся паттерны, зачем файл менялся).
- Если файл создаётся впервые (истории нет) — оценивай только корректность его кода.

Верни результат строго в формате JSON и ничего больше. Значения "verdict" оставь
латинскими (это фиксированный перечень), а "summary" и "text" пиши по-русски:
{
  "verdict": "approve | request_changes | comment",
  "summary": "краткое резюме ревью на русском",
  "comments": [
    {
      "file": "путь/к/файлу",
      "line": 42,
      "severity": "critical | major | minor | nit",
      "text": "текст комментария на русском"
    }
  ]
}
""",
    'review_prompt_greenplum.md': """Ты — старший инженер-ревьюер. Проанализируй изменения в pull request и оцени их
корректность.

Кодовая база — SQL-скрипты для Greenplum (версия 6.22, распределённая
PostgreSQL-совместимая MPP СУБД).

ПРАВИЛА ЯЗЫКА (строго, нарушение недопустимо):
- Отвечай ТОЛЬКО на русском языке. Все текстовые поля — "summary" и каждый
  "comments[].text" — должны быть на русском.
- Запрещено писать резюме или комментарии на английском (или любом другом языке),
  даже если diff, имена переменных или commit-сообщения на английском.
- Имена схем, таблиц, столбцов, функций и прочие технические идентификаторы
  оставляй без перевода — это код, а не текст.

ПРАВИЛА ПРОТИВ ГАЛЛЮЦИНАЦИЙ:
- Опирайся ТОЛЬКО на предоставленный diff и историю изменений. Не выдумывай код,
  названия схем/таблиц/полей или изменения, которых нет в diff.
- В поле "file" указывай только реальные пути из списка изменённых файлов.
- В поле "line" указывай только строки, реально присутствующие в diff; если точной
  строки нет — не заполняй "line".
- Если замечаний нет, верни пустой массив "comments": []. Не придумывай замечания,
  чтобы «что-то написать».
- Если по данным что-то не ясно — честно напиши об этом в "summary", не додумывай.

Критерии проверки (по убыванию важности):
1. Логические ошибки и баги.
2. Корректность SQL для Greenplum 6.22:
   - корректность ключа распределения (DISTRIBUTED BY / DISTRIBUTED RANDOMLY),
     отсутствие сильного перекоса данных (data skew);
   - правильный выбор типа таблицы (heap / append-optimized AO / AOCO) и
     партиционирования;
   - корректность внешних таблиц (gpfdist) и загрузки данных;
   - ограничения распределённого выполнения (недетерминированные функции,
     функции, выполняемые только на master).
3. Производительность: план запроса, join'ы крупных таблиц, агрегации, индексы.
4. Безопасность: SQL-инъекции, права доступа, работа с чувствительными данными.
5. Соответствие стилю и конвенциям кода.

Описание связанной задачи (work item):
[[WORK_ITEM]]

Изменённые файлы (полное содержимое + изменения + история предыдущих изменений):
[[FILES]]

Правила анализа:
- Если у файла есть история предыдущих изменений — учитывай её (сложившиеся паттерны, зачем файл менялся).
- Если файл создаётся впервые (истории нет) — оценивай только корректность его кода.

Верни результат строго в формате JSON и ничего больше. Значения "verdict" оставь
латинскими (это фиксированный перечень), а "summary" и "text" пиши по-русски:
{
  "verdict": "approve | request_changes | comment",
  "summary": "краткое резюме ревью на русском",
  "comments": [
    {
      "file": "путь/к/файлу",
      "line": 42,
      "severity": "critical | major | minor | nit",
      "text": "текст комментария на русском"
    }
  ]
}
""",
    'review_prompt_postgres.md': """Ты — старший инженер-ревьюер. Проанализируй изменения в pull request и оцени их
корректность.

Кодовая база — SQL-скрипты для PostgreSQL.

ПРАВИЛА ЯЗЫКА (строго, нарушение недопустимо):
- Отвечай ТОЛЬКО на русском языке. Все текстовые поля — "summary" и каждый
  "comments[].text" — должны быть на русском.
- Запрещено писать резюме или комментарии на английском (или любом другом языке),
  даже если diff, имена переменных или commit-сообщения на английском.
- Имена схем, таблиц, столбцов, функций и прочие технические идентификаторы
  оставляй без перевода — это код, а не текст.

ПРАВИЛА ПРОТИВ ГАЛЛЮЦИНАЦИЙ:
- Опирайся ТОЛЬКО на предоставленный diff и историю изменений. Не выдумывай код,
  названия схем/таблиц/полей или изменения, которых нет в diff.
- В поле "file" указывай только реальные пути из списка изменённых файлов.
- В поле "line" указывай только строки, реально присутствующие в diff; если точной
  строки нет — не заполняй "line".
- Если замечаний нет, верни пустой массив "comments": []. Не придумывай замечания,
  чтобы «что-то написать».
- Если по данным что-то не ясно — честно напиши об этом в "summary", не додумывай.

Критерии проверки (по убыванию важности):
1. Логические ошибки и баги.
2. Корректность SQL для PostgreSQL:
   - корректность DDL/DML, типов данных, ограничений (constraints);
   - корректность и идемпотентность миграций схемы (порядок применения);
   - корректная работа с транзакциями и конкурентным доступом.
3. Производительность: индексы, план запроса (EXPLAIN), блокировки, объём выборок.
4. Безопасность: SQL-инъекции, права доступа, работа с чувствительными данными.
5. Соответствие стилю и конвенциям кода.

Описание связанной задачи (work item):
[[WORK_ITEM]]

Изменённые файлы (полное содержимое + изменения + история предыдущих изменений):
[[FILES]]

Правила анализа:
- Если у файла есть история предыдущих изменений — учитывай её (сложившиеся паттерны, зачем файл менялся).
- Если файл создаётся впервые (истории нет) — оценивай только корректность его кода.

Верни результат строго в формате JSON и ничего больше. Значения "verdict" оставь
латинскими (это фиксированный перечень), а "summary" и "text" пиши по-русски:
{
  "verdict": "approve | request_changes | comment",
  "summary": "краткое резюме ревью на русском",
  "comments": [
    {
      "file": "путь/к/файлу",
      "line": 42,
      "severity": "critical | major | minor | nit",
      "text": "текст комментария на русском"
    }
  ]
}
""",
}

# render_prompt — дословная копия из prompt.py
def render_prompt(template, **values):
    """Подставляет плейсхолдеры [[КЛЮЧ]] значениями values (ключи — в верхнем регистре)."""
    result = template
    for key, value in values.items():
        result = result.replace(f"[[{key.upper()}]]", str(value))
    return result


def load_prompt(name: str = "review_prompt_greenplum.md") -> str:
    """В production: (prompts_dir / name).read_text(...). Здесь — из встроенного PROMPTS."""
    return PROMPTS[name]


print("Встроено production-промптов:", len(PROMPTS))
for k, v in PROMPTS.items():
    print(f"  {k:42s} {len(v):6d} симв.")

# Проверка подстановки плейсхолдеров
sample = render_prompt(
    PROMPTS["review_prompt_greenplum.md"],
    work_item="[Задача #1234] Подготовка витрины dwh.agr_sales",
    files="### Файл: /db/current/v_sales.sql (новый файл)\nselect ...",
)
print("\n--- render_prompt(): первые 300 симв. готового промпта ---")
print(sample[:300])

Встроено production-промптов: 5
  review_prompt_airflow_etl.md                 2727 симв.
  review_prompt_airflow_orchestration.md       2615 симв.
  review_prompt_clickhouse.md                  2604 симв.
  review_prompt_greenplum.md                   2726 симв.
  review_prompt_postgres.md                    2457 симв.

--- render_prompt(): первые 300 симв. готового промпта ---
Ты — старший инженер-ревьюер. Проанализируй изменения в pull request и оцени их
корректность.

Кодовая база — SQL-скрипты для Greenplum (версия 6.22, распределённая
PostgreSQL-совместимая MPP СУБД).

ПРАВИЛА ЯЗЫКА (строго, нарушение недопустимо):
- Отвечай ТОЛЬКО на русском языке. Все текстовые поля


### 3.3. Ядро ревью (production `review.py`)

Самая важная вставка: весь модуль `review.py`. Здесь собраны лимиты размера,
`_format_change`/`_build_file_section` (в промпт идёт и diff, и ПОЛНОЕ содержимое
файла + история 5 коммитов), `_chunk_sections` (жадное батчирование), толерантный
`_parse_json_response`, `_merge_results` (агрегация батчей), `_already_reviewed`
(dedup по маркеру) и функция `run_review` — оркестрация всего прогона.

In [7]:
# === Модуль review.py — ядро пайплайна (production-код, дословно) ===
# (убраны только относительные импорты: все определения уже есть в этом ноутбуке)

"""Основная функция ревью: единый проход по всем открытым PR с подробным логированием."""

import difflib
import json
import logging
from dataclasses import replace


logger = logging.getLogger("auto_git_review")

# Максимальный размер ПОЛНОГО содержимого одного файла, подаваемого в промпт
# (защита контекста ~30k токенов). Раньше для изменённых файлов подавался только
# unified diff с 3 строками контекста — модель видела фрагменты и считала код неполным.
MAX_CHANGE_CHARS = 12000

# Отдельный, меньший лимит на блок «изменённые строки» (diff) — вторичен по объёму.
MAX_DIFF_CHARS = 2000

# Максимальный суммарный размер одного «батча» файлов, подаваемого в ОДИН вызов LLM
# (~8–10k токенов). Большие PR ревьюятся по частям, результаты потом агрегируются.
MAX_BATCH_CHARS = 30000

CHANGE_TYPE_LABELS = {
    "add": "новый файл",
    "edit": "изменён",
    "delete": "удалён",
    "none": "без изменений",
}

# Маркер, по которому узнаём собственные комментарии бота (для dedup при повторных запусках).
COMMENT_MARKER = "auto_git_review"


def _parse_json_response(text: str):
    """Толерантный разбор JSON из ответа модели (модель может оборачивать в markdown)."""
    if not text:
        return None
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        if text.startswith("json"):
            text = text[4:]
        text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1 and end > start:
            try:
                return json.loads(text[start : end + 1])
            except json.JSONDecodeError:
                return None
        return None


def _truncate(text: str, limit: int = MAX_CHANGE_CHARS) -> str:
    if len(text) <= limit:
        return text
    return text[:limit] + "\n... (обрезано из-за ограничения контекста)"


def _describe_work_items(alm: AlmClient, pr_id: int) -> str:
    """Читаемое описание связанных work items PR (для подстановки в промпт)."""
    try:
        data = alm.get_pr_work_items(pr_id)
    except Exception as exc:
        logger.warning("Не удалось получить work items PR #%s: %s", pr_id, exc)
        return "(work items не найдены)"

    items = data.get("value", [])
    if not items:
        return "(work items не найдены)"

    descriptions = []
    for wi in items:
        wi_id = wi.get("id")
        try:
            detail = alm.get_work_item(wi_id)
            fields = detail.get("fields", {})
            wi_type = fields.get("System.WorkItemType", "?")
            wi_title = fields.get("System.Title", "?")
            descriptions.append(f"[{wi_type} #{wi_id}] {wi_title}")
        except Exception as exc:
            logger.warning("Не удалось получить work item #%s: %s", wi_id, exc)
            descriptions.append(f"[#{wi_id}]")
    return "\n".join(descriptions)


def _format_review_comment(parsed: dict) -> str:
    """Формирует markdown-текст общего комментария к PR из результата ревью."""
    verdict = parsed.get("verdict", "?")
    summary = (parsed.get("summary") or "").strip()
    comments = parsed.get("comments") or []

    lines = [f"**Автоматическое ревью ({COMMENT_MARKER})**", ""]
    lines.append(f"**Вердикт:** {verdict}")
    if summary:
        lines.append("")
        lines.append("**Резюме:**")
        lines.append(summary)
    if comments:
        lines.append("")
        lines.append("**Замечания:**")
        for c in comments:
            file_path = c.get("file", "")
            line = c.get("line")
            severity = c.get("severity", "")
            text = (c.get("text") or "").strip()
            locator = f"{file_path}:{line}" if line else file_path
            lines.append(f"- `{locator}` ({severity}): {text}")
    return "\n".join(lines)


def _already_reviewed(alm: AlmClient, pr_id: int) -> bool:
    """True, если у PR уже есть наш не-удалённый комментарий (по маркеру COMMENT_MARKER).

    Проверяем по маркеру в тексте, а не по автору: владелец PAT — реальный
    пользователь, у которого могут быть и собственные комментарии. Маркер
    однозначно отделяет комментарий бота.
    """
    try:
        threads = alm.get_threads(pr_id).get("value", [])
    except Exception as exc:
        # При ошибке проверки не можем гарантировать отсутствие дубля — пропускаем
        # постинг (лучше не отправить, чем отправить второй раз).
        logger.warning("Не удалось проверить комментарии PR #%s: %s — пропускаю отправку", pr_id, exc)
        return True

    for thread in threads:
        for comment in thread.get("comments", []):
            if comment.get("isDeleted"):
                continue
            if COMMENT_MARKER in (comment.get("content") or ""):
                return True
    return False


def _format_change(change: dict, new_content: str, old_content: str) -> str:
    """Читаемое представление одного изменения файла (unified diff или полное содержимое).

    new_content / old_content передаются сюда уже вытянутыми через alm.get_file_content()
    (endpoint iterations/{id}/changes контента не содержит).
    """
    change_type = change.get("changeType", "edit")
    path = (change.get("item") or {}).get("path", "?")

    if change_type == "add":
        return _truncate(f"[НОВЫЙ ФАЙЛ] {path}\n{new_content}")
    if change_type == "delete":
        return _truncate(f"[УДАЛЁН ФАЙЛ] {path}\n{old_content}")

    clean_path = path.lstrip("/")
    diff = difflib.unified_diff(
        old_content.splitlines(),
        new_content.splitlines(),
        fromfile=f"a/{clean_path}",
        tofile=f"b/{clean_path}",
        lineterm="",
    )
    diff_text = _truncate("\n".join(diff), MAX_DIFF_CHARS)
    # Показываем и изменённые строки, и ПОЛНОЕ новое содержимое файла: иначе модель
    # видит только фрагменты diff и считает, что «кода не хватает».
    return (
        f"[ИЗМЕНЁН] {path}\n\n"
        f"Изменённые строки (diff):\n{diff_text}\n\n"
        f"Полное содержимое файла (новая версия):\n{_truncate(new_content)}"
    )


def _format_history(commits: list) -> str:
    """Краткая история изменений файла: commit-сообщения."""
    if not commits:
        return "(истории нет — файл создаётся впервые)"
    lines = []
    for c in commits:
        sha = (c.get("commitId") or "")[:8]
        comment = (c.get("comment") or "").strip().splitlines()
        first = comment[0] if comment else "(без сообщения)"
        author = (c.get("author") or {}).get("name", "?")
        lines.append(f"- {sha} {first} (автор: {author})")
    return "\n".join(lines)


def _build_file_section(alm: AlmClient, change: dict, source_commit: str, target_commit: str) -> str:
    """Контекст одного изменённого файла: история + полное содержимое/изменения."""
    change_type = change.get("changeType", "edit")
    item = change.get("item") or {}
    path = item.get("path", "?")

    if change_type == "add":
        history_text = "(истории нет — файл создаётся впервые)"
    else:
        try:
            data = alm.get_file_commits(path, target_commit)
            history_text = _format_history(data.get("value", []))
        except Exception as exc:
            logger.warning("Не удалось получить историю файла %s: %s", path, exc)
            history_text = "(историю получить не удалось)"

    # Контент: для add — только новый, для delete — только старый, для edit — оба.
    new_content = ""
    old_content = ""
    try:
        if change_type in ("add", "edit"):
            new_content = alm.get_file_content(path, source_commit)
    except Exception as exc:
        logger.warning("Не удалось получить новый контент %s: %s", path, exc)
    try:
        if change_type in ("delete", "edit"):
            old_content = alm.get_file_content(path, target_commit)
    except Exception as exc:
        logger.warning("Не удалось получить старый контент %s: %s", path, exc)

    diff_text = _format_change(change, new_content, old_content)
    return (
        f"### Файл: {path} ({CHANGE_TYPE_LABELS.get(change_type, change_type)})\n"
        f"История предыдущих изменений:\n{history_text}\n\n"
        f"Текущее изменение:\n{diff_text}\n"
    )


def _build_file_sections(alm: AlmClient, changes: list, source_commit: str, target_commit: str) -> list:
    """Список секций контекста по изменённым файлам (tree-объекты пропускаются)."""
    sections = []
    for change in changes:
        item = change.get("item") or {}
        if item.get("gitObjectType") and item.get("gitObjectType") != "blob":
            continue
        sections.append(_build_file_section(alm, change, source_commit, target_commit))
    return sections


def _chunk_sections(sections: list, max_chars: int = MAX_BATCH_CHARS) -> list:
    """Жадно делит секции файлов на батчи, чтобы суммарный размер каждого ≤ max_chars."""
    batches = []
    current = []
    current_size = 0
    for section in sections:
        size = len(section)
        if current and current_size + size > max_chars:
            batches.append(current)
            current = []
            current_size = 0
        current.append(section)
        current_size += size
    if current:
        batches.append(current)
    return batches


VERDICT_PRIORITY = {"request_changes": 3, "comment": 2, "approve": 1}


def _merge_results(parsed_list: list) -> dict:
    """Агрегирует результаты ревью батчей в один итог (verdict + summary + comments)."""
    verdicts = [p.get("verdict", "approve") for p in parsed_list]
    verdict = max(verdicts, key=lambda v: VERDICT_PRIORITY.get(v, 0))

    summaries = []
    for i, p in enumerate(parsed_list, start=1):
        s = (p.get("summary") or "").strip()
        if s:
            summaries.append(s if len(parsed_list) == 1 else f"Часть {i}: {s}")
    summary = "\n\n".join(summaries)

    comments = []
    for p in parsed_list:
        comments.extend(p.get("comments") or [])
    return {"verdict": verdict, "summary": summary, "comments": comments}


def run_review(log=None, repo: str = None, project: str = None, post_comment: bool = False, prompt_name: str = None, max_batches: int = 20):
    """Главная функция: ревью всех открытых PR. Подробно логирует каждый шаг.

    Параметры (передаются из таски Airflow):
      - repo         — имя репозитория в ALM (если None — из настроек/config.py);
      - project      — имя проекта ALM (если None — из настроек/config.py);
      - post_comment — отправлять ли резюме комментарием в PR;
      - prompt_name  — файл промпта в prompts/ (для разных типов репозиториев);
      - max_batches  — максимум батчей на один PR; при превышении полный анализ
                       не выполняется, возвращается общее резюме.
    """
    log = log or logger

    log.info("=" * 60)
    log.info("СТАРТ: автоматическое ревью открытых PR")

    settings = get_settings()
    if repo:
        settings = replace(settings, azure_repo=repo)
    if project:
        settings = replace(settings, azure_project=project)
    if post_comment:
        settings = replace(settings, post_comments=True)
    prompt_name = prompt_name or "review_prompt_greenplum.md"

    log.info("Настройки загружены:")
    log.info("  ALM URL      : %s", settings.azure_url)
    log.info("  Проект       : %s", settings.azure_project)
    log.info("  Репозиторий  : %s", settings.azure_repo)
    log.info("  LLM модель   : %s", settings.llm_model)
    log.info("  LLM endpoint : %s", settings.llm_url)
    log.info("  Промпт       : %s", prompt_name)
    log.info("  Пост-коммент : %s", settings.post_comments)

    if not settings.azure_pat:
        raise RuntimeError("Не задан AZURE_DEVOPS_PAT (пустой токен).")
    if not settings.llm_api_key:
        raise RuntimeError("Не задан LLM_API_KEY (пустой ключ).")

    alm = AlmClient(settings)
    llm = LlmClient(settings)
    prompt_template = load_prompt(prompt_name)
    log.info("Промпт загружен из prompts/%s (%d символов)", prompt_name, len(prompt_template))

    log.info("Шаг 1: получаю список открытых PR из ALM...")
    prs = alm.list_open_pull_requests().get("value", [])
    log.info("Найдено открытых PR: %d", len(prs))

    if not prs:
        log.info("Нет открытых PR — ревью не требуется.")
        log.info("=" * 60)
        return []

    results = []
    for idx, pr in enumerate(prs, start=1):
        pr_id = pr["pullRequestId"]
        title = pr.get("title", "")
        author = pr.get("createdBy", {}).get("displayName", "?")
        log.info("-" * 60)
        log.info("PR %d/%d: #%s «%s» (автор: %s)", idx, len(prs), pr_id, title, author)
        log.info("  Ветки: %s -> %s", pr.get("sourceRefName"), pr.get("targetRefName"))

        # Dedup: проверяем наличие нашего комментария ДО запуска LLM — не тратим
        # вызовы модели на уже отревьюенные PR (только в режиме постинга).
        if settings.post_comments and _already_reviewed(alm, pr_id):
            log.info("    Комментарий от бота уже есть — пропускаю PR.")
            continue

        log.info("  Шаг 2: получаю детали PR #%s...", pr_id)
        detail = alm.get_pull_request(pr_id)
        src_commit = detail.get("lastMergeSourceCommit", {}).get("commitId")
        tgt_commit = detail.get("lastMergeTargetCommit", {}).get("commitId")
        log.info("    lastMergeSourceCommit: %s", src_commit)
        log.info("    lastMergeTargetCommit: %s", tgt_commit)

        log.info("  Шаг 3: получаю связанные work items...")
        work_items_text = _describe_work_items(alm, pr_id)
        log.info("    %s", work_items_text.replace("\n", "\n    "))

        log.info("  Шаг 4: собираю список изменённых файлов PR...")
        changes = []
        if not src_commit or not tgt_commit:
            log.warning("    Нет коммитов для diff, изменения не собраны.")
        else:
            try:
                iterations = alm.get_pr_iterations(pr_id).get("value", [])
                if iterations:
                    last_iteration = max(it["id"] for it in iterations)
                    changes = alm.get_pr_iteration_changes(pr_id, last_iteration).get(
                        "changeEntries", []
                    )
            except Exception as exc:
                log.warning("    Не удалось получить список файлов PR #%s: %s", pr_id, exc)
        log.info("    Изменённых файлов: %d", len(changes))

        log.info("  Шаг 5: собираю контекст файлов и делю на батчи...")
        sections = _build_file_sections(alm, changes, src_commit, tgt_commit)
        batches = _chunk_sections(sections)
        log.info("    Файлов: %d, батчей для LLM: %d", len(sections), len(batches))

        if not batches:
            log.warning("    Изменения не получены — ревью пропущено.")
            continue

        if len(batches) > max_batches:
            # Тяжёлый случай (напр. тысячи файлов): не гоняем сотни батчей через LLM,
            # формируем общее резюме с числом файлов и пометкой «нужно ручное ревью».
            log.warning(
                "    Слишком большой PR: %d файлов -> %d батчей (лимит %d) — полный анализ невозможен.",
                len(sections), len(batches), max_batches,
            )
            parsed = {
                "verdict": "comment",
                "summary": (
                    f"В PR изменено {len(sections)} файлов — объём слишком велик для "
                    f"полного автоматического анализа (превышен лимит батчей: "
                    f"{len(batches)} > {max_batches}). Требуется ручное ревью."
                ),
                "comments": [],
            }
        else:
            log.info("  Шаг 6: вызываю LLM (%s) по каждому батчу...", settings.llm_model)
            parsed_batches = []
            for bi, batch in enumerate(batches, start=1):
                files_text = "\n".join(batch)
                messages = [
                    {
                        "role": "user",
                        "content": render_prompt(
                            prompt_template,
                            work_item=work_items_text,
                            files=files_text,
                        ),
                    }
                ]
                try:
                    response = llm.chat(messages)
                    content = response["content"]
                except Exception as exc:
                    log.warning("    Батч %d/%d: ошибка вызова LLM: %s", bi, len(batches), exc)
                    continue
                log.info("    Батч %d/%d: ответ LLM (%d символов)", bi, len(batches), len(content))
                parsed = _parse_json_response(content)
                if parsed:
                    parsed_batches.append(parsed)
                else:
                    log.warning("    Батч %d/%d: не удалось разобрать JSON-ответ, сырой текст:", bi, len(batches))
                    log.warning("%s", content[:1000])

            log.info("  Шаг 7: агрегирую результаты батчей...")
            if not parsed_batches:
                log.warning("    Ни один батч не дал разобранный результат — ревью не сформировано.")
                results.append({"pr_id": pr_id, "verdict": "parse_error"})
                continue
            parsed = _merge_results(parsed_batches)

        verdict = parsed.get("verdict", "?")
        summary = parsed.get("summary", "")
        comments = parsed.get("comments", [])
        log.info("    Вердикт: %s", verdict)
        log.info("    Резюме: %s", summary)
        log.info("    Комментариев: %d", len(comments))

        if settings.post_comments:
            log.info("    Отправляю резюме в PR как комментарий...")
            try:
                comment_text = _format_review_comment(parsed)
                thread = alm.create_thread_comment(pr_id, comment_text)
                log.info("    Комментарий создан, thread id=%s", thread.get("id"))
            except Exception as exc:
                log.warning("    Не удалось отправить комментарий в PR #%s: %s", pr_id, exc)
        else:
            log.info("    Отправка комментариев отключена (POST_COMMENTS=false).")

        results.append({"pr_id": pr_id, "verdict": verdict, "comments": comments})

    log.info("=" * 60)
    log.info("ГОТОВО: обработано PR — %d", len(results))
    return results

print("review.py: ядро ревью готово.")

review.py: ядро ревью готово.


---
## 4. Гипотеза H1: dedup по маркеру не даёт повторного ревью

### 4.1. Проверка

Перед вызовом LLM `run_review` проверяет, есть ли у PR уже наш комментарий
(`_already_reviewed`). Критично: маркер ищется **в тексте** комментария, а не по
автору — владелец PAT реальный пользователь, у которого могут быть и собственные
комментарии. Удалённые комментарии не считаются. Ошибка проверки → «лучше не
отправить, чем отправить дважды» (`True`).

In [8]:
# === H1: сценарии (ground truth) для _already_reviewed ===
class _FakeAlm:
    def __init__(self, threads): self._threads = threads
    def get_threads(self, pr_id): return {"value": self._threads}

class _BrokenAlm:
    def get_threads(self, pr_id): raise RuntimeError("ALM недоступен")

def thread(content, deleted=False):
    return {"id": 0, "comments": [{"isDeleted": deleted, "content": content}]}

scenarios = [
    ("нет комментариев",              _FakeAlm([]),                              False),
    ("только комментарий человека",   _FakeAlm([thread("Смотрю код, ок.")]),      False),
    ("комментарий бота удалён",       _FakeAlm([thread("**Авторевью (auto_git_review)**", deleted=True)]), False),
    ("наш активный комментарий есть", _FakeAlm([thread("**Автоматическое ревью (auto_git_review)**")]), True),
    ("ошибка проверки ALM (fail-safe)", _BrokenAlm(),                             True),
]

print(f"{'Сценарий':38s} | {'_already_reviewed':>17s} | {'ожидание':>8s} | OK")
print("-" * 90)
all_ok = True
for name, fake, expected in scenarios:
    got = _already_reviewed(fake, pr_id=123)
    ok = got == expected
    all_ok &= ok
    print(f"{name:38s} | {str(got):>17s} | {str(expected):>8s} | {'✓' if ok else '✗'}")
print("\nВСЕ СЦЕНАРИИ ПРОЙДЕНЫ:", all_ok)

Не удалось проверить комментарии PR #123: ALM недоступен — пропускаю отправку


Сценарий                               | _already_reviewed | ожидание | OK
------------------------------------------------------------------------------------------
нет комментариев                       |             False |    False | ✓
только комментарий человека            |             False |    False | ✓
комментарий бота удалён                |             False |    False | ✓
наш активный комментарий есть          |              True |     True | ✓
ошибка проверки ALM (fail-safe)        |              True |     True | ✓

ВСЕ СЦЕНАРИИ ПРОЙДЕНЫ: True


### 4.2. Вывод по H1

**✓ Подтверждена.** Dedup по маркеру в тексте (а не по автору) корректно
отличает комментарий бота от комментариев реального пользователя, игнорирует
удалённые и при ошибке проверки уходит в fail-safe «не отправлять». Значит
повторные запуски ДАГа не тратят вызовы LLM и не плодят дубликаты.

---
## 5. Гипотеза H2: лимиты размера и батчирование защищают контекст LLM

### 5.1. Проверка

Контекст модели ограничен ~30k токенов. Защита тройная: (а) полное содержимое
одного файла обрезается до `MAX_CHANGE_CHARS`; (б) блок «изменённые строки»
(diff) — до `MAX_DIFF_CHARS`; (в) файлы жадно группируются в батчи ≤
`MAX_BATCH_CHARS`, каждый батч уходит в отдельный вызов LLM.

In [9]:
# === H2: синтетический набор секций и проверка батчей ===
# «Синтетический PR» — как synthetic.csv в similarity_search: известная структура.
def synth_sections(n, per_size=5400):
    secs = []
    for i in range(n):
        header = f"### Файл: /db/current/t{i:03d}.sql (изменён)\nИстория: (история есть)\n"
        body = "sql-data-select-" * (per_size // 18)
        secs.append(header + body + f" -- end {i}")
    return secs

sections = synth_sections(25)
batches = _chunk_sections(sections)

print(f"Файлов: {len(sections)} -> батчей: {len(batches)}")
for bi, b in enumerate(batches, 1):
    sz = sum(map(len, b))
    print(f"  батч {bi:2d}: {len(b):2d} файлов, {sz:6d} символов  (лимит {MAX_BATCH_CHARS})")
assert all(sum(map(len, b)) <= MAX_BATCH_CHARS for b in batches), "батч превысил лимит!"
print("OK: ни один батч не превышает MAX_BATCH_CHARS.")

# (а) обрезка ОДНОГО гигантского файла
giant = "-- header\n" + "select 1; -- padding\n" * 4000          # ~ 76k симв.
trunc = _truncate(giant)
print(f"\n(а) гигантский файл {len(giant)} симв. -> после _truncate {len(trunc)} симв.")
assert len(trunc) <= MAX_CHANGE_CHARS + 100
print("OK: однофайловый лимит соблюдён, добавлена пометка об обрезке.")

# (б) для edit в промпт идут И diff, И полное содержимое
chg = {"changeType": "edit", "item": {"path": "/db/x.sql"}}
block = _format_change(chg, "select a, b from t where id > 0;\n", "select a from t;\n")
print("\n(б) пример блока _format_change (edit):")
print(block[:300])
assert "Изменённые строки (diff)" in block and "Полное содержимое файла" in block
print("OK: модель получает и изменённые строки, и весь файл.")

Файлов: 25 -> батчей: 5
  батч  1:  6 файлов,  29244 символов  (лимит 30000)
  батч  2:  6 файлов,  29246 символов  (лимит 30000)
  батч  3:  6 файлов,  29250 символов  (лимит 30000)
  батч  4:  6 файлов,  29250 символов  (лимит 30000)
  батч  5:  1 файлов,   4875 символов  (лимит 30000)
OK: ни один батч не превышает MAX_BATCH_CHARS.

(а) гигантский файл 84010 симв. -> после _truncate 12043 симв.
OK: однофайловый лимит соблюдён, добавлена пометка об обрезке.

(б) пример блока _format_change (edit):
[ИЗМЕНЁН] /db/x.sql

Изменённые строки (diff):
--- a/db/x.sql
+++ b/db/x.sql
@@ -1 +1 @@
-select a from t;
+select a, b from t where id > 0;

Полное содержимое файла (новая версия):
select a, b from t where id > 0;

OK: модель получает и изменённые строки, и весь файл.


### 5.2. Вывод по H2

**✓ Подтверждена.** Секции файлов (diff + полное содержимое + история) жадно
группируются так, что каждый батч не превышает `MAX_BATCH_CHARS`; отдельный
слишком большой файл обрезается с явной пометкой. Модель никогда не получает
запрос больше лимита контекста — ревью больших PR идёт «частями».

---
## 6. Гипотеза H3: приоритетная агрегация и толерантный JSON-парсер

### 6.1. Проверка

Вердикт всего PR — это «худший» из вердиктов батчей по приоритету
`request_changes > comment > approve`; summary склеивается с пометкой «Часть N»;
замечания конкатенируются. Ответы модели могут приходить в markdown-обёртке —
`_parse_json_response` их снимает.

In [10]:
# === H3: агрегация вердиктов и устойчивость парсинга ===
batches_cases = [
    (["approve"],                                                 "approve"),
    (["approve", "comment"],                                      "comment"),
    (["comment", "request_changes"],                              "request_changes"),
    (["approve", "request_changes", "comment"],                   "request_changes"),
]
print("Вердикты батчей -> итог:")
all_ok = True
for verdicts, expected in batches_cases:
    merged = _merge_results([{"verdict": v, "summary": "", "comments": []} for v in verdicts])
    ok = merged["verdict"] == expected
    all_ok &= ok
    print(f"  {verdicts} -> {merged['verdict']:16s} (ожид. {expected:16s}) {'✓' if ok else '✗'}")

# summary склейка «Часть N» и конкатенация комментариев
merged = _merge_results([
    {"verdict": "comment", "summary": "Батч 1 ок", "comments": [{"file": "a.sql", "line": 1, "text": "x"}]},
    {"verdict": "comment", "summary": "Есть риск", "comments": [{"file": "b.sql", "line": 5, "text": "y"}]},
])
print("\nsummary после склейки:\n", merged["summary"])
print("комментариев после склейки:", len(merged["comments"]))
assert len(merged["comments"]) == 2 and "Часть 2" in merged["summary"]

# толерантный разбор JSON (markdown-обёртка и мусор вокруг)
dirty = '```json\n{"verdict": "comment", "summary": "Проверить индексы", "comments": []}\n```'
parsed = _parse_json_response(dirty)
print("\n_parse_json_response из markdown-обёртки:", parsed)
assert parsed and parsed["verdict"] == "comment"
print("OK: JSON-ответ модели разбирается даже с ```-обёрткой.")

Вердикты батчей -> итог:
  ['approve'] -> approve          (ожид. approve         ) ✓
  ['approve', 'comment'] -> comment          (ожид. comment         ) ✓
  ['comment', 'request_changes'] -> request_changes  (ожид. request_changes ) ✓
  ['approve', 'request_changes', 'comment'] -> request_changes  (ожид. request_changes ) ✓

summary после склейки:
 Часть 1: Батч 1 ок

Часть 2: Есть риск
комментариев после склейки: 2

_parse_json_response из markdown-обёртки: {'verdict': 'comment', 'summary': 'Проверить индексы', 'comments': []}
OK: JSON-ответ модели разбирается даже с ```-обёрткой.


### 6.2. Вывод по H3

**✓ Подтверждена.** Итоговый вердикт соответствует пессимистичному приоритету,
summary батчей склеивается с нумерацией «Часть N», замечания не теряются.
Ответы LLM с ```` ```json ````-обёрткой (и мусором) разбираются без падения
пайплайна.

---
## 7. Гипотеза H4: лимит батчей защищает от «гигантских» PR

### 7.1. Проверка

Если PR порождает больше `MAX_BATCHES = 20` батчей (например, тысячи файлов),
полный прогон через LLM не выполняется: вместо сотен дорогих вызовов формируется
общее резюме с числом файлов и пометкой «требуется ручное ревью». Это ветка
`run_review` из `review.py`.

In [11]:
# === H4: ветка run_review при превышении MAX_BATCHES ===
def oversize_fallback(sections, max_batches=MAX_BATCHES):
    """Срез ветки review.run_review (строки ~381-396) — до вызова LLM."""
    batches = _chunk_sections(sections)
    if len(batches) <= max_batches:
        return None, batches
    parsed = {
        "verdict": "comment",
        "summary": (
            f"В PR изменено {len(sections)} файлов — объём слишком велик для полного "
            f"автоматического анализа (превышен лимит батчей: {len(batches)} > {max_batches}). "
            f"Требуется ручное ревью."
        ),
        "comments": [],
    }
    return parsed, batches

# «Гигантский» PR: 400 файлов по ~5.4k симв. -> ~75 батчей (заведомо больше лимита)
giant_sections = synth_sections(400)
parsed, batches = oversize_fallback(giant_sections)
print(f"Файлов: {len(giant_sections)} -> батчей: {len(batches)} (лимит {MAX_BATCHES})")
print(f"Вердикт (без вызова LLM): {parsed['verdict']}")
print(parsed["summary"])
assert parsed["verdict"] == "comment" and "Требуется ручное ревью" in parsed["summary"]
assert len(batches) > MAX_BATCHES

# Обратная сторона: обычный PR проходит на LLM (батчей <= лимита)
normal_sections = synth_sections(20)
parsed, batches = oversize_fallback(normal_sections)
print(f"\nОбычный PR: {len(normal_sections)} файлов -> {len(batches)} батчей, LLM-вызов возможен")
assert parsed is None and len(batches) <= MAX_BATCHES
print("OK: обычный PR не триггерит защиту, гигантский — возвращает резюме без LLM.")

Файлов: 400 -> батчей: 67 (лимит 20)
Вердикт (без вызова LLM): comment
В PR изменено 400 файлов — объём слишком велик для полного автоматического анализа (превышен лимит батчей: 67 > 20). Требуется ручное ревью.

Обычный PR: 20 файлов -> 4 батчей, LLM-вызов возможен
OK: обычный PR не триггерит защиту, гигантский — возвращает резюме без LLM.


### 7.2. Вывод по H4

**✓ Подтверждена.** Порог `MAX_BATCHES` не даёт «сжечь» десятки LLM-вызовов на
PR из тысяч файлов: до модели доходит только общее резюме с требованием ручного
ревью.

---
## 8. Применение к реальным данным

### 8.1. Живой прогон на одном PR (без постинга)

Повторяем тело `run_review` для одного открытого PR выбранного репозитория: сбор
изменений из последней итерации, контекст файлов, батчирование, вызовы LLM по
батчам, агрегация и формирование markdown-комментария. **Комментарий в PR не
постится** — он только печатается (это ровно тот текст, который бот отправил бы
в ALM). Требуется `.env` с корпоративными ключами и доступ к контуру.

In [12]:
# === 8.2 Живой прогон: один PR, post_comments=False ===
try:
    require_live()
    repo_cfg = REPOSITORIES[0]                       # Greenplum ExampleProject1
    project, repo, kind, prompt_name = repo_cfg
    alm = AlmClient(settings)
    llm = LlmClient(settings)
    prompt_tpl = load_prompt(prompt_name)

    # выбираем первый открытый PR без нашего маркера
    prs = [p for p in alm.list_open_pull_requests(repo=repo).get("value", [])
           if not _already_reviewed(alm, p["pullRequestId"])]
    print(f"[{project}/{repo}] открытых PR без нашего комментария: {len(prs)}")
    if not prs:
        print("Демонстрация остановлена: нет подходящего PR.")
        raise SystemExit

    pr = prs[0]
    pr_id = pr["pullRequestId"]
    print(f"PR #{pr_id} «{pr.get('title','')}» ({pr.get('createdBy',{}).get('displayName','?')})")

    detail = alm.get_pull_request(pr_id, repo=repo)
    src = detail.get("lastMergeSourceCommit", {}).get("commitId")
    tgt = detail.get("lastMergeTargetCommit", {}).get("commitId")
    wis = _describe_work_items(alm, pr_id)

    iterations = alm.get_pr_iterations(pr_id, repo=repo).get("value", [])
    last_it = max(it["id"] for it in iterations)
    changes = alm.get_pr_iteration_changes(pr_id, last_it, repo=repo).get("changeEntries", [])

    sections = _build_file_sections(alm, changes, src, tgt)
    batches = _chunk_sections(sections)
    print(f"Файлов: {len(sections)}, батчей для LLM: {len(batches)}")

    if len(batches) > MAX_BATCHES:
        parsed = {
            "verdict": "comment",
            "summary": f"В PR изменено {len(sections)} файлов — объём слишком велик "
                       f"(батчей {len(batches)} > {MAX_BATCHES}). Требуется ручное ревью.",
            "comments": [],
        }
    else:
        parsed_batches = []
        for bi, batch in enumerate(batches, 1):
            msg = [{"role": "user", "content": render_prompt(
                prompt_tpl, work_item=wis, files="\n".join(batch))}]
            content = llm.chat(msg)["content"]
            parsed = _parse_json_response(content)
            if parsed:
                parsed_batches.append(parsed)
            print(f"  батч {bi}/{len(batches)}: ответ LLM ({len(content)} симв.) -> "
                  f"verdict={parsed.get('verdict') if parsed else 'parse_error'}")
        if not parsed_batches:
            print("Ни один батч не дал разобранный результат — ревью не сформировано.")
            raise SystemExit
        parsed = _merge_results(parsed_batches)

    print(f"\n=== Вердикт: {parsed.get('verdict')} ===\n")
    print(_format_review_comment(parsed))
    print("\n(комментарий НЕ отправлен в PR — только демонстрация текста)")
except SystemExit:
    pass
except Exception as exc:
    print(f"SKIP 8.2 (живой прогон): {type(exc).__name__}: {exc}")

SKIP 8.2 (живой прогон): RuntimeError: Нет доступа к живым эндпоинтам: отсутствуют AZURE_DEVOPS_PAT, LLM_API_KEY. Заполните .env из .env.example и запустите в корпоративной сети.


### 8.3. Что даёт живой прогон

При наличии доступа клетка выше показывает весь пайплайн «в бою» на настоящем PR:
список изменённых файлов из последней итерации, контекст (diff + полное
содержимое + история 5 коммитов), батчи, реальные ответы `qwen3:latest` и итоговый
markdown-комментарий. Повторный запуск ДАГа не дублирует комментарий благодаря H1
(маркер уже присутствует в PR).

---
## 9. Заключение

### 9.1. Сводка по гипотезам

| № | Гипотеза | Статус |
|---|---|---|
| H1 | Dedup по маркеру до вызова LLM не даёт повторного ревью | ✓ Подтверждена (раздел 4) |
| H2 | Батчирование и обрезки держат каждый вызов LLM в лимите контекста | ✓ Подтверждена (раздел 5) |
| H3 | Приоритетная агрегация вердиктов и толерантный JSON-парсер корректны | ✓ Подтверждена (раздел 6) |
| H4 | Лимит `MAX_BATCHES` защищает от «гигантских» PR без прогона через LLM | ✓ Подтверждена (раздел 7) |

Живая проверка на реальных данных — в **разделе 8** (требует корпоративного
доступа). Инварианты H1–H4 проверены детерминированно на синтетических
сценариях (как `synthetic.csv` в `similarity_search`).

### 9.2. Применимость

- Мульти-проектность/мульти-репозиторность: одна схема на 6 репозиториев в двух
  проектах ALM, свой промпт под тип кодовой базы (Greenplum, ClickHouse,
  PostgreSQL, Airflow).
- Всё работает внутри корпоративного периметра: локальный ALM + локальная LLM.
- Открытые вопросы / Фаза 2: запуск по расписанию и через Service Hook (webhook),
  построчные комментарии к файлам вместо одного общего резюме.